# 03 · Clasificación del resultado de una orden de servicio

Esta libreta vuelve a ejecutar el pipeline académico de clasificación y deja el mismo artefacto JSON que el backend usa para predecir si una orden terminará a tiempo, retrasada o cancelada.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess

BACKEND = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'package.json').exists() and (path / 'src').exists()
)
npm = 'npm.cmd' if os.name == 'nt' else 'npm'
result = subprocess.run(
    [npm, 'run', 'classification:train'],
    cwd=BACKEND,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)


In [ ]:
csv_path = BACKEND / 'ml' / 'classification' / 'data' / 'classification_service_outcomes.csv'
artifact_path = BACKEND / 'ml' / 'classification' / 'artifacts' / 'classification_service_outcome_model.json'
report_path = BACKEND / 'ml' / 'classification' / 'reports' / 'classification_training_report.json'
artifact = json.loads(artifact_path.read_text(encoding='utf-8'))
report = json.loads(report_path.read_text(encoding='utf-8'))
csv_hash = hashlib.sha256(csv_path.read_bytes()).hexdigest()
assert csv_hash == artifact['datasetSha256'] == report['datasetSha256']

print('Artefacto usado por backend:', artifact_path)
print('Training samples:', artifact['dataset']['trainingSamples'])
print('Test samples:', artifact['dataset']['evaluationTestSamples'])
print('Accuracy:', artifact['evaluation']['accuracy'])
print('Macro F1:', artifact['evaluation']['macroAverage']['f1Score'])


El backend carga este JSON una sola vez y reutiliza sus pesos y preprocesamiento para inferencia; la libreta y la aplicación comparten exactamente el mismo artefacto.